## Resultados del Detector

El `PatternDetector` retorna un objeto `PatternDetectionResult` que contiene:

- Lista de todos los patrones detectados con sus scores
- El patron principal identificado
- Conteo de patrones
- Metadata del analisis

### Flujo de Deteccion

```
AST -> Detectores -> PatternMatches -> Scorer -> PatternDetectionResult
 |         |              |              |              |
 |    [DivConq,       [Match1,      [Scored1,     { patterns,
 |     DP, ...]        Match2]       Scored2]      primary,
 |                                                 count }
```

### Propiedades del Resultado

| Propiedad | Tipo | Descripcion |
|-----------|------|-------------|
| all_patterns | List[ScoredPattern] | Todos los patrones rankeados |
| primary_pattern | ScoredPattern | Patron con mayor score |
| pattern_count | int | Total de patrones detectados |
| detection_timestamp | datetime | Momento del analisis |

## Estructura PatternMatch

Cada patron detectado se representa como `PatternMatch`:

```python
@dataclass
class PatternMatch:
    pattern_type: PatternType     # Tipo enum del patron
    pattern_name: str             # Nombre legible
    confidence: float             # Confianza (0.0 - 1.0)
    confidence_level: ConfidenceLevel  # Nivel categorico
    indicators_found: List[str]   # Indicadores detectados
    indicators_missing: List[str] # Indicadores no encontrados
    description: str              # Descripcion del patron
    complexity_hint: str          # Complejidad tipica
    supporting_evidence: Dict     # Evidencia adicional
```

### Tipos de Patron (PatternType)

```python
class PatternType(str, Enum):
    BRUTE_FORCE = "brute_force"
    RECURSIVE = "recursive"
    DIVIDE_AND_CONQUER = "divide_and_conquer"
    DYNAMIC_PROGRAMMING = "dynamic_programming"
    GREEDY = "greedy"
    BACKTRACKING = "backtracking"
    BRANCH_AND_BOUND = "branch_and_bound"
    SORTING = "sorting"
    SEARCHING = "searching"
    UNKNOWN = "unknown"
```

### Ejemplo de PatternMatch

```python
PatternMatch(
    pattern_type=PatternType.DIVIDE_AND_CONQUER,
    pattern_name="Divide y Venceras",
    confidence=0.88,
    confidence_level=ConfidenceLevel.HIGH,
    indicators_found=[
        "calculo_punto_medio",
        "llamadas_recursivas_multiples",
        "caso_base"
    ],
    indicators_missing=["fase_combinacion_explicita"],
    description="Patron que divide el problema en subproblemas...",
    complexity_hint="T(n) = aT(n/b) + f(n)",
    supporting_evidence={
        "recursive_calls": 2,
        "division_factor": 2
    }
)
```

## PatternDetectionResult

Resultado completo de la deteccion:

```python
@dataclass
class PatternDetectionResult:
    all_patterns: List[ScoredPattern]
    primary_pattern: Optional[ScoredPattern]
    pattern_count: int
    detection_timestamp: datetime
    analysis_duration_ms: float
    metadata: Dict[str, Any]
```

### Metodos Utiles

```python
# Obtener patrones por tipo
result.get_patterns_by_type(PatternType.RECURSIVE)

# Filtrar por confianza minima
result.filter_by_confidence(0.70)

# Verificar si un patron especifico fue detectado
result.has_pattern(PatternType.DYNAMIC_PROGRAMMING)

# Obtener el patron principal
primary = result.primary_pattern
```

### Conversion a Diccionario

Para serializar el resultado:

```python
result_dict = result.to_dict()
# {
#     "patterns": [...],
#     "primary": {...},
#     "count": 3,
#     "timestamp": "2024-...",
#     "duration_ms": 15.2
# }
```

## Integracion con Analisis de Complejidad

Los patrones detectados complementan el analisis de complejidad:

### AnalyzerEngine

El motor de analisis integra ambos sistemas:

```python
from app.core.analyzer.analyzer_engine import AnalyzerEngine

engine = AnalyzerEngine()
result = engine.analyze(ast)

# Acceder a complejidad
print(result.complexity.big_o)  # O(n log n)

# Acceder a patrones
print(result.patterns.primary_pattern)  # Divide y Venceras
```

### Correlacion Complejidad-Patron

| Patron | Complejidad Tipica |
|--------|--------------------|
| Divide y Venceras | O(n log n), O(n^2) |
| Programacion Dinamica | O(n), O(n^2), O(2^n) |
| Voraz | O(n log n), O(n) |
| Backtracking | O(n!), O(2^n) |
| Fuerza Bruta | O(n^2), O(n^3) |
| Busqueda Binaria | O(log n) |

### Validacion Cruzada

El sistema puede validar que la complejidad calculada sea coherente
con el patron detectado:

```python
# Alerta si complejidad no coincide
if pattern == DIVIDE_AND_CONQUER and "log n" not in complexity:
    warnings.append("Posible discrepancia en analisis")
```

## Validacion de Resultados

### Metricas de Evaluacion

Para evaluar la precision del detector:

| Metrica | Formula | Descripcion |
|---------|---------|-------------|
| Precision | VP / (VP + FP) | Exactitud de detecciones |
| Recall | VP / (VP + FN) | Cobertura de patrones |
| F1-Score | 2 * (P * R) / (P + R) | Balance precision-recall |

Donde:
- VP = Verdaderos Positivos (patron correcto detectado)
- FP = Falsos Positivos (patron incorrecto detectado)
- FN = Falsos Negativos (patron correcto no detectado)

### Dataset de Validacion

Conjunto de algoritmos con patrones conocidos:

| Algoritmo | Patron Esperado | Notas |
|-----------|-----------------|-------|
| Merge Sort | Divide y Venceras | Divisiones recursivas |
| Fibonacci DP | Prog. Dinamica | Tabla de memorizacion |
| Selection Sort | Fuerza Bruta, Ordenamiento | Bucles anidados |
| Binary Search | Busqueda, Divide y Venceras | Division iterativa |
| Knapsack DP | Prog. Dinamica | Tabla 2D |
| N-Queens | Backtracking | Prueba y retroceso |
| Huffman | Voraz | Seleccion optima local |

### Proceso de Validacion

1. **Parsear** algoritmo de prueba
2. **Detectar** patrones automaticamente
3. **Comparar** con patron esperado
4. **Registrar** resultado (acierto/fallo)
5. **Calcular** metricas globales

## Demostracion: Evaluacion de Deteccion

In [ ]:
import sys
sys.path.insert(0, '../..')

from app.core.parser.pseudocode_parser import PseudocodeParser
from app.core.patterns.pattern_detector import PatternDetector

parser = PseudocodeParser()
detector = PatternDetector()

# Casos de prueba
casos = [
    {
        "nombre": "Binary Search",
        "esperado": ["searching", "divide_and_conquer"],
        "codigo": '''
algorithm binarySearch(A[], x, low, high)
begin
    if (low > high) then
        return -1
    end
    mid <- (low + high) / 2
    if (A[mid] = x) then
        return mid
    else
        if (A[mid] > x) then
            return call binarySearch(A, x, low, mid - 1)
        else
            return call binarySearch(A, x, mid + 1, high)
        end
    end
end
'''
    },
    {
        "nombre": "Bubble Sort",
        "esperado": ["sorting", "brute_force"],
        "codigo": '''
algorithm bubbleSort(A[], n)
begin
    for i <- 0 to n - 1 do
        for j <- 0 to n - i - 1 do
            if (A[j] > A[j + 1]) then
                temp <- A[j]
                A[j] <- A[j + 1]
                A[j + 1] <- temp
            end
        end
    end
end
'''
    }
]

print("=== Evaluacion de Deteccion de Patrones ===")
print()

for caso in casos:
    print(f"Algoritmo: {caso['nombre']}")
    print(f"Patrones esperados: {caso['esperado']}")
    
    ast = parser.parse(caso['codigo'])
    resultado = detector.detect(ast)
    
    detectados = [p.pattern.pattern_type.value for p in resultado.all_patterns]
    print(f"Patrones detectados: {detectados}")
    
    # Verificar aciertos
    aciertos = set(caso['esperado']) & set(detectados)
    faltantes = set(caso['esperado']) - set(detectados)
    extra = set(detectados) - set(caso['esperado'])
    
    print(f"Aciertos: {list(aciertos)}")
    if faltantes:
        print(f"Faltantes: {list(faltantes)}")
    if extra:
        print(f"Adicionales: {list(extra)}")
    print("-" * 40)

## Analisis de Indicadores Detectados

In [ ]:
# Analisis detallado de indicadores

codigo_merge_sort = '''
algorithm mergeSort(A[], left, right)
begin
    if (left < right) then
        mid <- (left + right) / 2
        call mergeSort(A, left, mid)
        call mergeSort(A, mid + 1, right)
        call merge(A, left, mid, right)
    end
end
'''

ast = parser.parse(codigo_merge_sort)
resultado = detector.detect(ast)

print("=== Analisis Detallado: Merge Sort ===")
print()

for sp in resultado.all_patterns:
    pm = sp.pattern
    print(f"Patron: {pm.pattern_name}")
    print(f"Confianza: {pm.confidence:.2%} ({pm.confidence_level.value})")
    print(f"Score Final: {sp.final_score:.2%}")
    print()
    print("Indicadores encontrados:")
    for ind in pm.indicators_found:
        print(f"  + {ind}")
    print()
    if pm.indicators_missing:
        print("Indicadores faltantes:")
        for ind in pm.indicators_missing:
            print(f"  - {ind}")
        print()
    print(f"Complejidad tipica: {pm.complexity_hint}")
    print("=" * 50)

## Interpretacion de Resultados

### Casos Exitosos

Un resultado exitoso presenta:

- Patron primario con confianza >= 0.70
- Indicadores principales encontrados
- Complejidad coherente con el patron

### Casos Ambiguos

Cuando multiples patrones tienen confianza similar:

```
Patron A: 0.72
Patron B: 0.68
```

Considerar:
- El algoritmo puede implementar multiples patrones
- Los indicadores se solapan entre patrones
- Revisar indicadores especificos de cada uno

### Casos de Baja Confianza

Cuando todos los patrones tienen confianza < 0.50:

- El algoritmo puede no seguir patrones conocidos
- Puede requerir nuevos indicadores
- Considerar clasificacion "desconocido"

### Recomendaciones

| Situacion | Accion |
|-----------|--------|
| Alta confianza unico | Reportar patron primario |
| Alta confianza multiples | Reportar todos los relevantes |
| Confianza media | Incluir advertencia |
| Baja confianza | Marcar como tentativo |

---

## Conclusiones

El sistema de evaluacion de patrones permite:

1. **Analizar detalladamente** cada patron detectado
2. **Verificar indicadores** encontrados y faltantes
3. **Comparar** con resultados esperados
4. **Medir precision** del detector
5. **Integrar** con analisis de complejidad

### Flujo Completo de Uso

```
1. Parsear codigo          -> AST
2. Detectar patrones       -> PatternDetectionResult
3. Analizar complejidad    -> ComplexityResult
4. Integrar resultados     -> AnalysisResult completo
5. Evaluar y reportar      -> Informe final
```

---

**Este notebook completa la serie de documentacion del sistema de deteccion de patrones.**

Para continuar explorando, consulte:
- `01_exploratory/` para experimentar con el parser
- `02_complexity_analysis/` para analisis de complejidad